# 06 · Strong students: full fine-tune of DenseOn, Granite R2, E5-base-v2 (DDP, 2 GPUs)
Three ~110–150M retrievers replace the Qwen student. Each one gets a **full fine-tune** (every weight + a linear head
768→256, no LoRA, no pruning) with the L2 alignment loss: put the student's query vector on the teacher's vector of
the same query, L2 = ||s − t||² = 2 − 2·cos. The documents stay the frozen teacher `corpus.npy`.

| row | hub id | arch | layers / hidden | pooling | query prompt |
|---|---|---|---|---|---|
| `denseon_aligned` | `lightonai/DenseOn` | ModernBERT | 22 / 768 | CLS | `"query: "` |
| `granite_r2_aligned` | `ibm-granite/granite-embedding-english-r2` | ModernBERT | 22 / 768 | CLS | none |
| `e5_base_aligned` | `intfloat/e5-base-v2` | BERT | 12 / 768 | mean | `"query: "` |

Pooling is read from each model's own `1_Pooling/config.json` (`model.read_pooling`), not set here.
**Early stopping**: `patience=3` stops a run after 3 evals (every `eval_every` steps) without a val_loss drop; the
best weights become `final/`. Every rank prints its GPU peak / free memory after every eval.

Input: `01`. Output (attach to `04`): `runs/<mode>/{denseon,granite_r2,e5_base}_aligned/`, rows in `runs/<mode>/metrics.json`.

In [ ]:
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("../..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}


## Control panel

In [ ]:
import os
import shutil
import torch
from model import get_device
from teacher import TEACHER_DEFAULTS as T

DEBUG = False    # use the same value as in 01
NPROCS = 2

# --- paths. On Kaggle, earlier notebooks' outputs are attached as inputs (read only); set the slugs to yours.
#     Off Kaggle everything lives under the repo, so every IN_* is the same dir.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/datasets/tensorpanda231"
IN_01 = f"{KAGGLE_IN}/corpus-teacher-embd" if ON_KAGGLE else REPO
DATA_PATH = f"{IN_01}/cache/{MODE}/data.json"
EMB_DIR = f"{IN_01}/cache/{MODE}/emb/teacher"
RUN_DIR = f"{ROOT}/runs/{MODE}"
PREV_METRICS = f"{IN_01}/runs/{MODE}/metrics.json"

# --- students: name -> ALIGN overrides. query_prefix = the model's own query prompt (same in training and EVAL).
STUDENTS = {
    "denseon":    dict(model="lightonai/DenseOn", query_prefix="query: "),
    "granite_r2": dict(model="ibm-granite/granite-embedding-english-r2", query_prefix=""),
    "e5_base":    dict(model="intfloat/e5-base-v2", query_prefix="query: "),
}

# --- alignment, full fine-tune, shared by every student (align.ALIGN_DEFAULTS; unknown keys raise)
ALIGN = dict(full=True,                           # every base weight + head trains; no LoRA (lora_* unused)
             dim=T["dim"],                        # head output = teacher index dim
             max_query_len=32,                    # includes the prompt tokens
             loss="l2", loss_kwargs={},           # ||s - t||^2 = 2 - 2 cos between unit vectors
             lr=7e-5,                             # all base weights (bge-base value)
             head_lr=3e-4,                        # the head starts random
             weight_decay=0.0, warmup_ratio=0.05,
             epochs=12, batch_size=256, grad_accum=1,  # per GPU; global batch = batch_size * NPROCS * grad_accum
             grad_clip=1.0, amp=True, seed=0,
             log_every=20, eval_every=100,
             patience=3,                          # evals without a val_loss drop -> stop, best weights -> final
             save_every=400, keep_last=1,
             data_path=DATA_PATH, teacher_emb_dir=EMB_DIR)

# --- evaluation (evaluation.evaluate_model kwargs; query_prefix is set per student): student queries vs the
#     frozen teacher corpus, top 100
EVAL = dict(splits=("val", "test"), dim=T["dim"], max_query_len=32, device=get_device(), dtype="fp16")
LATENCY = dict(n_queries=2000, batch_sizes=(1, 32))

if DEBUG:
    ALIGN.update(log_every=5, eval_every=20, save_every=20)
    LATENCY.update(n_queries=200)

## 1 · Check inputs
Teacher vectors valid (finite, unit norm, `data.json` order) for the corpus, val/test and the alignment queries.

In [ ]:
from data import load_emb, load_json
from evaluation import check_emb, load_index

# rows accumulate across notebooks: start from the previous notebook's metrics.json (copied once)
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(f"{RUN_DIR}/metrics.json") and os.path.exists(PREV_METRICS):
    shutil.copy(PREV_METRICS, f"{RUN_DIR}/metrics.json")
data = load_json(DATA_PATH)
index = load_index(data, EMB_DIR, ("val", "test"))   # (teacher docs, {split: teacher queries})
emb, ids = load_emb(EMB_DIR, "align")
check_emb(emb, ids, data["align"]["ids"], "align")
steps = len(ids) // (ALIGN["batch_size"] * NPROCS * ALIGN["grad_accum"])
print(f"align {emb.shape}: {steps} steps/epoch, at most {steps * ALIGN['epochs']} steps per student")

## 2 · Load check (one GPU, in this process)
For each student: load it, print its architecture / pooling / padding and the tokens of one prefixed query, encode
64 val queries (finite, unit norm), then one fp16 forward + backward of the L2 loss at the per-GPU `batch_size`
(what each DDP rank does per micro-batch). Every weight must get a gradient (an unused one would hang DDP). The
peak memory plus the AdamW state (8 bytes per param) is roughly what one rank needs.

In [ ]:
import transformers
from align import l2_loss
from model import attach_head, embed, encode, gpu_mem, load_encoder, tokenize
from teacher import free_gpu_memory

dev = get_device()
print(f"transformers {transformers.__version__}, torch {torch.__version__}, {dev}")
align_texts, align_t = data["align"]["texts"], torch.from_numpy(emb.astype("float32"))
bs, max_len, dim = ALIGN["batch_size"], ALIGN["max_query_len"], ALIGN["dim"]
for name, s in STUDENTS.items():
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats(dev)
    model, tok = load_encoder(s["model"], dev)
    c = model.config
    n_params = sum(p.numel() for p in model.parameters())
    print(f"--- {name} ({s['model']}): {c.model_type}, {c.num_hidden_layers} layers, hidden {c.hidden_size}, "
          f"{n_params / 1e6:.0f}M params, pooling {model.pooling}, padding {tok.padding_side}")
    enc = tokenize(tok, data["val"]["texts"][:1], max_len, s["query_prefix"])
    print("    tokens:", tok.convert_ids_to_tokens(enc["input_ids"][0]))
    q = encode(model, tok, data["val"]["texts"][:64], dev, dim, max_len, s["query_prefix"]).float()
    assert torch.isfinite(q).all() and torch.allclose(q.norm(dim=1), torch.ones(len(q)), atol=1e-2), name

    attach_head(model, dim)
    model.requires_grad_(True).train()
    enc = tokenize(tok, align_texts[:bs], max_len, s["query_prefix"])
    with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
        student = embed(model, enc["input_ids"].to(dev), enc["attention_mask"].to(dev), dim)
    loss = l2_loss(student.float(), align_t[:bs].to(dev))
    loss.backward()
    no_grad = [k for k, p in model.named_parameters() if p.grad is None]
    assert not no_grad, f"{name}: no gradient for {no_grad[:5]}"
    line = f"    untrained L2 {loss.item():.3f}, every weight has a gradient"
    if dev.type == "cuda":
        line += (f"; fwd+bwd at batch {bs}: peak {torch.cuda.max_memory_allocated(dev) / 2**30:.1f} GB allocated "
                 f"+ {8 * n_params / 2**30:.1f} GB AdamW state in training{gpu_mem(dev)}")
    print(line)
    del model, student, loss
    free_gpu_memory()

## 3 · Align every student (full fine-tune)
Rank r takes its slice of every global batch; DDP averages the gradients of every weight. Val L2 / cos / nDCG@10 are
logged every `eval_every` steps, followed by each rank's GPU peak memory; early stopping on val_loss
(`patience`). A finished student (`final/` exists) is skipped, an interrupted one resumes from its last step
checkpoint.

In [ ]:
from align import align_main
from run_exp import train_once

runs = {}
for name, s in STUDENTS.items():
    cfg = {**ALIGN, **s, "out_dir": f"{RUN_DIR}/{name}_aligned"}
    print(f"===== {name}: {s['model']}")
    runs[f"{name}_aligned"] = train_once(align_main, cfg, NPROCS)

## 4 · Training curves
`train_loss` and `val_loss` are both L2, comparable across students.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from run_exp import plot_curves, save_fig

fig = plot_curves({n: f"{RUN_DIR}/{n}" for n in runs}, ("train_loss", "val_loss", "val_ndcg@10"), "strong students")
display(save_fig(fig, RUN_DIR, "strong_students_curves"))
plt.close("all")

## 5 · Retrieval quality (val + test)
Student queries (each with its own prompt) vs the frozen teacher corpus. retention = nDCG@10 / teacher nDCG@10.

In [ ]:
from run_exp import evaluate_runs, print_rows

for name, s in STUDENTS.items():
    metrics = evaluate_runs({f"{name}_aligned": runs[f"{name}_aligned"]}, data, index, RUN_DIR,
                            {**EVAL, "query_prefix": s["query_prefix"]})
for split in ("val", "test"):
    print_rows(metrics, list(runs), split)
    print()

## 6 · Query latency (one GPU, fp16)

In [ ]:
from evaluation import measure_latency

texts = data["val"]["texts"][:LATENCY["n_queries"]]
for name, s in STUDENTS.items():
    model, tok = load_encoder(runs[f"{name}_aligned"], EVAL["device"], torch.float16)
    print(f"--- {name}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
    measure_latency(model, tok, texts, EVAL["device"], T["dim"], EVAL["max_query_len"], s["query_prefix"],
                    LATENCY["batch_sizes"])
    del model
    free_gpu_memory()